In [31]:

!pip install pandas

In [34]:

import pandas as pd
df = pd.read_csv("../data/contract_evalution_dataset_bezwada-krishna-chaitanya.csv")
df.head()

,contract_text,expected_apr,expected_term,expected_payment,expected_penalty
0,This Car Loan Agreement offers an APR of 9.5% ...,9.50,60.0,18500,Late payment penalty Rs. 500
1,The loan carries an annual percentage rate (AP...,11.00,48.0,21200,NaN
2,APR applicable is 8.75%. The duration of this ...,8.75,36.0,24000,2% overdue penalty
3,This agreement has a tenure of 72 months with ...,NaN,72.0,16300,Late fee Rs. 750
4,APR is fixed at 10.25% per annum. The borrower...,10.25,60.0,19100,NaN


In [51]:
EXPECTED_OUTPUT_SCHEMA = dict.fromkeys(
    ["apr", "term_months", "monthly_payment", "penalty_clause"],
    None
)


In [ ]:
#Promt template for contract evaluation
PROMPT_TEMPLATE =""""
Extract the following feilds:
{{"apr":"",
"term_months":"",
"expected_payment":"",
"penalty_clause":""
}}

contract:
{{contract_text}}
"""

In [53]:
#Extract function
def extract_contract_fields(contract_text, llm_client):
    prompt = PROMPT_TEMPLATE.format(contract_text=contract_text)
    llm_response = llm_client(prompt)
    return json.loads(llm_response)


In [54]:
#dummy llm client for testing
def dummy_llm_client(_):
    return json.dumps(EXPECTED_OUTPUT_SCHEMA)


In [55]:
#Sample data
sample_df = df.sample(n=5, random_state=42)


In [85]:
#Run Extraction on sample data and Collecting results
records = []

for _, row in sample_df.iterrows():
    output = extract_contract_fields(
        row["contract_text"],
        dummy_llm_client
    )

    records.append({
        "contract_text": row["contract_text"],
        "llm_apr": output.get("apr"),
        "llm_term": output.get("term_months"),
        "llm_payment": output.get("monthly_payment"),
        "llm_penalty": output.get("penalty_clause"),
        "expected_apr": row["expected_apr"],
        "expected_term": row["expected_term"],
        "expected_payment": row["expected_payment"],
        "expected_penalty": row["expected_penalty"]
    })

results_df = pd.DataFrame(records)

In [83]:
print(df.columns)


Index(['contract_text', 'expected_apr', 'expected_term', 'expected_payment',
       'expected_penalty'],
      dtype='object')


In [57]:
#exctatly matching results
def exact_match(a, b):
    return int(str(a).strip() == str(b).strip())


In [86]:
#evaluate results
match_columns = {
    "apr_match": ("llm_apr", "expected_apr"),
    "term_match": ("llm_term", "expected_term"),
    "payment_match": ("llm_payment", "expected_payment"),
    "penalty_match": ("llm_penalty", "expected_penalty"),
}

for col, (llm_col, exp_col) in match_columns.items():
    results_df[col] = results_df.apply(
        lambda r: exact_match(r[llm_col], r[exp_col]),
        axis=1
    )
results_df.head()

,contract_text,llm_apr,llm_term,llm_payment,llm_penalty,expected_apr,expected_term,expected_payment,expected_penalty,apr_match,term_match,payment_match,penalty_match
0,"The borrower agrees to pay Rs. 20,000 per mont...",None,None,None,None,9.0,48.0,20000,Late penalty Rs. 600,0,0,0,0
1,APR fixed at 6.9% for a term of 24 months. Mon...,None,None,None,None,6.9,24.0,34500,NaN,0,0,0,0
2,This Car Loan Agreement offers an APR of 9.5% ...,None,None,None,None,9.5,60.0,18500,Late payment penalty Rs. 500,0,0,0,0
3,This car loan agreement mentions term of 72 mo...,None,None,None,None,NaN,72.0,16800,NaN,0,0,0,0
4,"APR: 8.5%. Monthly payment Rs. 22,700. Term no...",None,None,None,None,8.5,NaN,22700,NaN,0,0,0,0
